## Exploration of training data
Notes:

- Where would data augmentation be most useful? 
    - "we should make dates more varied:
    - "we should make sure there is a greater representation of answers that are labeled as '4'"


Below, we look at:
- label (0/1) distribution
- gpt_score distribution
- Distribution of date format in responses
- Distribution of number format in responses

In [13]:
import pandas as pd

In [14]:
# MultiRC With Reference Answer and GPT5 4-point scores:
multirc_path = "../../data/multirc-data-w-gpt5-scores-v3.csv"

# matching column names in notebook 5
df = pd.read_csv(multirc_path, index_col=0)
df

,provided_split,passage,question,response,label,passage_id,question_id,answer_id,answer,gpt5_score
split,,,,,,,,,,
train,train,"The rally took place on October 17, the shooti...",When was Kayla Rolland shot?,February 17,0,1,26,168,She was shot on February 29.,1
train,train,"The rally took place on October 17, the shooti...",When was Kayla Rolland shot?,February 29,1,1,26,169,She was shot on February 29.,3
train,train,"The rally took place on October 17, the shooti...",When was Kayla Rolland shot?,October 29,0,1,26,170,She was shot on February 29.,1
train,train,"The rally took place on October 17, the shooti...",When was Kayla Rolland shot?,October 17,0,1,26,171,She was shot on February 29.,1
train,train,"The rally took place on October 17, the shooti...",When was Kayla Rolland shot?,February 17,0,1,26,172,She was shot on February 29.,1
...,...,...,...,...,...,...,...,...,...,...
test,validation,You have seen your own reflection in a mirror....,What is similar to your reflection?,The same image as you,0,79,917,4669,The image of the sign above (the sign’s reflec...,2
test,validation,You have seen your own reflection in a mirror....,What is similar to your reflection?,Your image is reversed and looks just like you,1,79,917,4670,The image of the sign above (the sign’s reflec...,2
test,validation,You have seen your own reflection in a mirror....,What causes the image in a mirror reflection t...,Light rays strike flat shiny surfaces and are ...,1,79,918,4671,"Because light rays strike the flat, shiny surf...",2


In [46]:
# 1. Label distribution analysis

# 'label' column
print('=' * 50)
print("'label' distribution (0=incorrect, 1=correct)")
print('=' * 50)

label_counts = df['label'].value_counts().sort_index()
print("Counts:")
for label, count in label_counts.items():
    proportion = count / len(df)
    print(f"Label {label}: {count:4d} ({proportion:6.1%})")

# 'gpt5_score' column
print()
print('=' * 50)
print("'gpt5_score' distribution (1, 2, 3, 4)")
print("=" * 50)

score_counts = df['gpt5_score'].value_counts().sort_index()
print("Counts:")
for score, count in score_counts.items():
    proportion = count / len(df)
    print(f"Score {score}: {count:4d} ({proportion:6.1%})")

# RESULTS: 
# - label 0 and 1 are pretty fairly distributed --> no data augmentation needed
# - gpt5_score 2 is underrepresented (15.7%) and 4 is extremely underrepresented (4.8%)
#      --> maybe generated 2-3x more responses for gpt5_score=4 and 1.5x more responses for gpt5_score=2

'label' distribution (0=incorrect, 1=correct)
Counts:
Label 0: 17991 ( 56.1%)
Label 1: 14100 ( 43.9%)

'gpt5_score' distribution (1, 2, 3, 4)
Counts:
Score 1: 15390 ( 48.0%)
Score 2: 5024 ( 15.7%)
Score 3: 10123 ( 31.5%)
Score 4: 1554 (  4.8%)


In [47]:
# 2. Response pattern analysis
#  - includes dates, number format

def analyze_format_diversity(df, column, patterns, category_name):
    """
    Analyze format diversity for a specific pattern category (dates, numbers, etc.)
    
    Args:
        df: DataFrame to analyze
        column: Column name to search in (e.g., 'response')
        patterns: Dict of {pattern_name: regex_pattern}
        category_name: Name for this category (e.g., 'Date', 'Numeric')
        threshold_high: Proportion threshold for high priority (default 0.90)
        threshold_medium: Proportion threshold for medium priority (default 0.75)
    """
    print("=" * 50)
    print(f"{category_name} patterns in responses")
    print("=" * 50)
    
    # Track which format each response matches
    format_matches = {pattern_name: [] for pattern_name in patterns.keys()}
    
    for pattern_name, pattern in patterns.items():
        matches = df[df[column].str.contains(pattern, case=False, na=False, regex=True)]
        format_matches[pattern_name] = matches.index.tolist()
        if len(matches) > 0:
            print(f"{pattern_name}: {len(matches)} matches")
            print(f"Examples: {matches[column].head(10).tolist()}\n")
    
    # Proportions analysis
    print("\n" + "=" * 50)
    print(f"{category_name.upper()} format distribution")
    print("=" * 50)
    
    format_counts = {name: len(indices) for name, indices in format_matches.items()}
    total_matches = sum(format_counts.values())
    
    if total_matches > 0:
        print(f"Format counts:")
        for pattern_name, count in sorted(format_counts.items(), key=lambda x: x[1], reverse=True):
            proportion = count / total_matches
            print(f"{pattern_name:20s}: {count:4d} ({proportion:6.1%})")
    return format_matches, format_counts

In [48]:
# 2.1 Date pattern detection:
# - find if there a specific format that a majority of date answers fall under.
#   - if so, we this might mean we need to make dates more varied

date_patterns = {
    'Month DD': r'\b(?:January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{1,2}\b',
    'Mon DD': r'\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Sept|Oct|Nov|Dec)\.?\s+\d{1,2}',
    'MM/DD': r'\b\d{1,2}/\d{1,2}\b',
    'DD Month': r'\b\d{1,2}\s+(?:January|February|March|April|May|June|July|August|September|October|November|December)\b',
}

date_matches, date_counts = analyze_format_diversity(df, 'response', date_patterns, 'Date')

# RESULTS:
# - Month DD makes up 70.4% of the date responses
#   --> other formats (Mon DD, DD Month, MM/DD) should be generated

Date patterns in responses
Month DD: 95 matches
Examples: ['February 17', 'February 29', 'October 29', 'October 17', 'February 17', 'The after-action review had treated the CIA as the lead agency for any offensive against al Qaeda, and the principals, at their March 10 meeting, had endorsed strengthening the CIA\'s capability for that role. To the CTC, that meant proceeding with "The Plan," which it had put forward half a year earlier-hiring and training more case officers and building up the capabilities of foreign security services that provided intelligence via liaison', 'August 15', 'August 12', 'August 15', 'August 15, 1928']

Mon DD: 24 matches
Examples: ['Aug 6', 'Aug. 15,', 'Aug 6', 'Aug 15', 'Sept 11', 'Requirements are any plan for the site replace all the office, hotel and retail square footage lost in the attack on Sept. 11', 'On Dec 20th', 'On Dec 20th', 'By late May 2000, two operatives assigned to the planes operation were already in the United States. Three of the four 

In [49]:
# 2.2 Numeric pattern detection:
# - similar to date: find if one particular pattern (ex. digit only) makes up the majority of responses

numeric_patterns = {
    'Digit only': r'^\d+$',
    'Digit with unit': r'^\d+\s+\w+',
    'Spelled out (standalone)': r'^(?:one|two|three|four|five|six|seven|eight|nine|ten|eleven|twelve|thirteen|fourteen|fifteen|sixteen|seventeen|eighteen|nineteen|twenty|thirty|forty|fifty|sixty|seventy|eighty|ninety|hundred|thousand)$',
    'Spelled out (with unit)': r'^(?:one|two|three|four|five|six|seven|eight|nine|ten|eleven|twelve|thirteen|fourteen|fifteen|sixteen|seventeen|eighteen|nineteen|twenty|thirty|forty|fifty|sixty|seventy|eighty|ninety|hundred|thousand)\s+\w+',
}

numeric_matches, numeric_counts = analyze_format_diversity(df, 'response', numeric_patterns, 'Numeric')

# RESULTS:
# - digit only makes up 45%, while spelled out makes up 7.0%
#   --> generate more spelled-out variants

# Note - spelled out category contains answers like "The young one" and "No one knows," which is included because the word "one" matched but 
# is not relevant in comparing to the other two patterns

Numeric patterns in responses
Digit only: 932 matches
Examples: ['2', '3', '1', '2', '8', '16', '10', '6', '1997', '1996']

Digit with unit: 746 matches
Examples: ['10 days', '5 years', '22 years had passed', '20 years had passed', '22 years', '22 years from 1883 to 1905', '1 year', '10 years', '7 years', '18 years']

Spelled out (standalone): 145 matches
Examples: ['Two', 'Three', 'One', 'Five', 'Three', 'Six', 'Ten', 'Twelve', 'Eleven', 'Ten']

Spelled out (with unit): 250 matches
Examples: ['Two parts namely tire and sun gatherer', 'One year', 'Two children pushing a swing, book resting on a table', 'Two forces acting on it', 'Two grants totally over $6.5 million dollars', 'One time', 'Three times', 'Two main features - Continents and Ocean Basins', 'Thousand years old', 'Eight Banks']


NUMERIC format distribution
Format counts:
Digit only          :  932 ( 45.0%)
Digit with unit     :  746 ( 36.0%)
Spelled out (with unit):  250 ( 12.1%)
Spelled out (standalone):  145 (  7.0%)


In [50]:
# 2.3 Capitalization Patterns

print("=" * 50)
print("Capitalization Patterns")
print("=" * 50)

cap_analysis = df.copy()
cap_analysis['is_title_case'] = cap_analysis['response'].str.istitle()
cap_analysis['is_upper'] = cap_analysis['response'].str.isupper()
cap_analysis['is_lower'] = cap_analysis['response'].str.islower()
cap_analysis['is_mixed'] = ~(cap_analysis['is_title_case'] | cap_analysis['is_upper'] | cap_analysis['is_lower'])

print(f"Title case: {cap_analysis['is_title_case'].sum()} ({cap_analysis['is_title_case'].sum()/len(df)*100:.1f}%)")
print(f"All uppercase: {cap_analysis['is_upper'].sum()} ({cap_analysis['is_upper'].sum()/len(df)*100:.1f}%)")
print(f"All lowercase: {cap_analysis['is_lower'].sum()} ({cap_analysis['is_lower'].sum()/len(df)*100:.1f}%)")
print(f"Mixed case: {cap_analysis['is_mixed'].sum()} ({cap_analysis['is_mixed'].sum()/len(df)*100:.1f}%)")

# RESULTS
# - reasonable diversity across responses.
# - all uppercase representation is very low (0.8%), but these types of responses aren't expected anyways (?)
#   --> no action needed

Capitalization Patterns
Title case: 10176 (31.7%)
All uppercase: 267 (0.8%)
All lowercase: 1360 (4.2%)
Mixed case: 20323 (63.3%)


In [51]:
# 2.4 Response Length Analysis
print("=" * 50)
print("Response Length Distribution")
print("=" * 50)

df['response_length'] = df['response'].str.len()
df['response_words'] = df['response'].str.split().str.len()

print(f"\nCharacter length - Mean: {df['response_length'].mean():.1f}, Median: {df['response_length'].median()}")
print(f"Word count - Mean: {df['response_words'].mean():.1f}, Median: {df['response_words'].median()}")

# By label
print("\nLength by binary label:")
print(df.groupby('label')['response_length'].describe())

# By gpt5_score
print("\nLength by GPT5 score:")
print(df.groupby('gpt5_score')['response_length'].describe())

# RESULTS:
# - label 1 responses are longer than label 0 responses by an average of 20 characters (~4 words)
# - gpt5_score 4 responses are longer than nonpassing scores, 1 and 2, by 60 and 47 characters, respectively
#   --> generate shorter paraphrases for high scores (especially 4s)
#   --> generate longer paraphrases for low scores (especially 1s)

Response Length Distribution

Character length - Mean: 26.7, Median: 16.0
Word count - Mean: 4.7, Median: 3.0

Length by binary label:
         count       mean        std  min   25%   50%   75%    max
label                                                             
0      17981.0  19.229464  17.907650  1.0   8.0  14.0  25.0  224.0
1      14095.0  36.166016  40.385679  1.0  11.0  22.0  47.0  453.0

Length by GPT5 score:
              count       mean        std  min   25%   50%    75%    max
gpt5_score                                                              
1           15375.0  17.869463  16.674247  1.0   7.0  13.0   23.0  208.0
2            5024.0  29.273487  29.272261  1.0  11.0  20.0   37.0  367.0
3           10123.0  30.972439  33.585807  1.0  10.0  20.0   40.0  430.0
4            1554.0  77.334620  60.169154  4.0  34.0  63.0  103.0  453.0


### RESULTS:

Analyzed: 
- label distribution (0/1) 
- gpt5_score distribution (1, 2, 3, 4)
- date format patterns
- digit/numeric response patterns
- capitalization patterns
- response length patterns

Labels/gpt5_score distribution
- label 0 and 1 are pretty fairly distributed --> no data augmentation needed
- gpt5_score 2 is underrepresented (15.7%) and 4 is extremely underrepresented (4.8%) --> **maybe generate 2-3x more responses for gpt5_score=4 and 1.5x more responses for gpt5_score=2**

Date format patterns
- Month DD makes up 70.4% of the date responses --> **other formats (Mon DD, DD Month, MM/DD) should be generated**

Digit/numeric response patterns
- digit only makes up 45%, while spelled out makes up 7.0% --> **generate more spelled-out variants**

Capitalization patterns
- Wasn't sure if this was really an issue/relevant in this case
- reasonable diversity across responses (all lowercase, mixed case).
- all uppercase representation is very low (0.8%), but these types of responses aren't expected anyways (?) --> no action needed

Response length
- label 1 responses are longer than label 0 responses by an average of 20 characters (~4 words)
- gpt5_score 4 responses are longer than nonpassing scores, 1 and 2, by 60 and 47 characters, respectively --> **generate shorter paraphrases for high scores (especially 4s), generate longer paraphrases for low scores (especially 1s)**